# Time-Based Train / Validation Split

## Purpose

TRACE is a fraud-detection system, so validation should represent the real situation:

```text
Past transactions → TRAIN
Future transactions → VALIDATION
```

A time split prevents future transactions from being used to learn the training period.


In [67]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "extracted_data"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)


Project root: C:\Users\Manojit Khatua\AI ENGINEERING\trace-ai
Data path: C:\Users\Manojit Khatua\AI ENGINEERING\trace-ai\data\processed\extracted_data


## 1. Load the transaction data

`train_transaction.csv` contains the transaction-level data and the `isFraud` target.


In [3]:
train_transaction = pd.read_csv(
    DATA_PATH / "train_transaction.csv"
)

print("Dataset shape:", train_transaction.shape)


Dataset shape: (590540, 394)


## 2. Verify the temporal column

`TransactionDT` is used only to preserve chronological order. We do not convert it to calendar time in this notebook.


In [4]:
if "TransactionDT" not in train_transaction.columns:
    raise ValueError("TransactionDT column is missing.")

if train_transaction["TransactionDT"].isna().any():
    raise ValueError("TransactionDT contains missing values.")

print("Minimum TransactionDT:", train_transaction["TransactionDT"].min())
print("Maximum TransactionDT:", train_transaction["TransactionDT"].max())


Minimum TransactionDT: 86400
Maximum TransactionDT: 15811131


## 3. Sort the transactions chronologically

The earliest transactions must come first so that the later split represents future data.


In [5]:
train_transaction = (
    train_transaction
    .sort_values("TransactionDT")
    .reset_index(drop=True)
)

print(
    "Chronological order:",
    train_transaction["TransactionDT"].is_monotonic_increasing
)


Chronological order: True


## 4. Create the chronological split

We use an 80/20 split by time:

- First 80% → training
- Last 20% → validation


In [6]:
train_ratio = 0.80
split_index = int(len(train_transaction) * train_ratio)

train_data = train_transaction.iloc[:split_index].copy()
validation_data = train_transaction.iloc[split_index:].copy()

print("Train shape:", train_data.shape)
print("Validation shape:", validation_data.shape)


Train shape: (472432, 394)
Validation shape: (118108, 394)


## 5. Verify the time boundaries

The latest training transaction must occur before the earliest validation transaction.


In [7]:
train_min = train_data["TransactionDT"].min()
train_max = train_data["TransactionDT"].max()
validation_min = validation_data["TransactionDT"].min()
validation_max = validation_data["TransactionDT"].max()

print("TRAIN")
print("Min:", train_min)
print("Max:", train_max)

print("\nVALIDATION")
print("Min:", validation_min)
print("Max:", validation_max)


TRAIN
Min: 86400
Max: 12192842

VALIDATION
Min: 12192900
Max: 15811131


In [8]:
if train_max > validation_min:
    raise ValueError("Temporal overlap detected.")

print("Time split is valid: training data comes before validation data.")


Time split is valid: training data comes before validation data.


## 6. Check fraud distribution

Fraud prevalence can change over time, so we compare the target distribution in both periods.


In [9]:
train_fraud_rate = train_data["isFraud"].mean()
validation_fraud_rate = validation_data["isFraud"].mean()

print("Train fraud rate:", train_fraud_rate)
print("Validation fraud rate:", validation_fraud_rate)

print("Train fraud count:", int(train_data["isFraud"].sum()))
print("Validation fraud count:", int(validation_data["isFraud"].sum()))


Train fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145
Train fraud count: 16599
Validation fraud count: 4064


## 7. Attach identity information after the split

`DeviceInfo` and `DeviceType` are stored in `train_identity.csv`, not in the transaction table.

We attach them **after** the time split using `TransactionID`.

This keeps the chronological split intact while making identity information available for later relationship features.


In [10]:
train_identity = pd.read_csv(
    DATA_PATH / "train_identity.csv"
)

print("Identity shape:", train_identity.shape)


Identity shape: (144233, 41)


In [11]:
identity_columns = [
    "TransactionID",
    "DeviceInfo",
    "DeviceType"
]

train_data = train_data.merge(
    train_identity[identity_columns],
    on="TransactionID",
    how="left"
)

validation_data = validation_data.merge(
    train_identity[identity_columns],
    on="TransactionID",
    how="left"
)


## 8. Verify the identity merge

The merge must not change the number of transactions.

```text
TRAIN       → same number of rows
VALIDATION  → same number of rows
```


In [12]:
print("DeviceInfo in train:", "DeviceInfo" in train_data.columns)
print("DeviceInfo in validation:", "DeviceInfo" in validation_data.columns)

print("Train rows:", len(train_data))
print("Validation rows:", len(validation_data))


DeviceInfo in train: True
DeviceInfo in validation: True
Train rows: 472432
Validation rows: 118108


In [13]:
expected_train_rows = split_index
expected_validation_rows = len(train_transaction) - split_index

if len(train_data) != expected_train_rows:
    raise ValueError("Train row count changed after identity merge.")

if len(validation_data) != expected_validation_rows:
    raise ValueError("Validation row count changed after identity merge.")

print("Identity merge validation passed.")


Identity merge validation passed.


# Entity mappings from training

In [14]:
entity_columns = ["card1", "card2", "addr1"]

entity_maps = {
    column: train_data[column].value_counts(dropna=False)
    for column in entity_columns}
for column, mapping in entity_maps.items():
    print(f"{column}: {len(mapping)} unique values")

card1: 12730 unique values
card2: 501 unique values
addr1: 329 unique values


In [16]:
for column, mapping in entity_maps.items():
    feature_name = f"{column}_transaction_count"

    train_data[feature_name] = (
        train_data[column].map(mapping).fillna(0).astype("int32")
    )

    validation_data[feature_name] = (
        validation_data[column].map(mapping).fillna(0).astype("int32")
    )

In [17]:
train_data[["card1","card1_transaction_count","card2","card2_transaction_count","addr1","addr1_transaction_count"]].head()

,card1,card1_transaction_count,card2,card2_transaction_count,addr1,addr1_transaction_count
0,13926,42,NaN,7025,315.0,18435
1,2755,570,404.0,2584,325.0,34061
2,4663,886,490.0,30168,330.0,19826
3,18132,3383,567.0,4925,476.0,7651
4,4497,13,514.0,11793,420.0,2853


In [19]:
validation_data[["card1","card1_transaction_count","card2","card2_transaction_count","addr1","addr1_transaction_count"]].head()

,card1,card1_transaction_count,card2,card2_transaction_count,addr1,addr1_transaction_count
0,9300,997,103.0,2907,NaN,53761
1,8809,44,179.0,83,NaN,53761
2,10819,1,555.0,33820,NaN,53761
3,9633,3520,130.0,1984,NaN,53761
4,17188,8214,321.0,38743,310.0,6711


# card1 + DeviceInfo relationship

In [27]:
pair_data = train_data.dropna(subset=["DeviceInfo"])

pair_map = (pair_data.groupby(["card1", "DeviceInfo"]).size())

In [28]:
train_pairs = pd.MultiIndex.from_frame(train_data[["card1", "DeviceInfo"]])
validation_pairs = pd.MultiIndex.from_frame(validation_data[["card1", "DeviceInfo"]])
train_data["card1_DeviceInfo_transaction_count"] = (train_pairs.map(pair_map).fillna(0).astype("int32"))
validation_data["card1_DeviceInfo_transaction_count"] = (validation_pairs.map(pair_map).fillna(0).astype("int32"))

In [29]:
train_data[["card1", "DeviceInfo", "card1_DeviceInfo_transaction_count"]].head(10)

,card1,DeviceInfo,card1_DeviceInfo_transaction_count
0,13926,NaN,0
1,2755,NaN,0
2,4663,NaN,0
3,18132,NaN,0
4,4497,SAMSUNG SM-G892A Build/NRD90M,1
5,5937,NaN,0
6,12308,NaN,0
7,12695,NaN,0
8,2803,iOS Device,199
9,17399,NaN,0


In [30]:
validation_data[["card1", "DeviceInfo", "card1_DeviceInfo_transaction_count"]].head(10)

,card1,DeviceInfo,card1_DeviceInfo_transaction_count
0,9300,NaN,0
1,8809,F3311,0
2,10819,Windows,0
3,9633,Windows,884
4,17188,Windows,255
5,9026,Windows,537
6,6200,NaN,0
7,17188,NaN,0
8,12695,NaN,0
9,16132,NaN,0


In [48]:
card_device_map = (
    train_data.dropna(subset=["DeviceInfo"])
    .groupby("card1")["DeviceInfo"]
    .nunique()
)

print("Card1 mappings:", len(card_device_map))

Card1 mappings: 7354


In [49]:
train_data["card1_unique_DeviceInfo_count"] = (
    train_data["card1"]
    .map(card_device_map)
    .fillna(0)
    .astype("int32")
)

validation_data["card1_unique_DeviceInfo_count"] = (
    validation_data["card1"]
    .map(card_device_map)
    .fillna(0)
    .astype("int32")
)

In [50]:
train_data[
    ["card1", "DeviceInfo", "card1_unique_DeviceInfo_count"]
].head(10)

,card1,DeviceInfo,card1_unique_DeviceInfo_count
0,13926,NaN,4
1,2755,NaN,13
2,4663,NaN,4
3,18132,NaN,34
4,4497,SAMSUNG SM-G892A Build/NRD90M,3
5,5937,NaN,0
6,12308,NaN,2
7,12695,NaN,33
8,2803,iOS Device,52
9,17399,NaN,9


In [51]:
validation_data[
    ["card1", "DeviceInfo", "card1_unique_DeviceInfo_count"]
].head(10)

,card1,DeviceInfo,card1_unique_DeviceInfo_count
0,9300,NaN,173
1,8809,F3311,12
2,10819,Windows,0
3,9633,Windows,343
4,17188,Windows,65
5,9026,Windows,164
6,6200,NaN,0
7,17188,NaN,65
8,12695,NaN,33
9,16132,NaN,37


# Learn the mapping from TRAIN

In [52]:
device_card_map = (train_data.dropna(subset=["DeviceInfo"]).groupby("DeviceInfo")["card1"].nunique())

In [53]:
train_data["DeviceInfo_unique_card1_count"] = (train_data["DeviceInfo"].map(device_card_map).fillna(0).astype("int32"))

validation_data["DeviceInfo_unique_card1_count"] = (validation_data["DeviceInfo"].map(device_card_map).fillna(0).astype("int32"))

In [54]:
train_data[
    ["DeviceInfo", "card1", "DeviceInfo_unique_card1_count"]
].head(10)

,DeviceInfo,card1,DeviceInfo_unique_card1_count
0,NaN,13926,0
1,NaN,2755,0
2,NaN,4663,0
3,NaN,18132,0
4,SAMSUNG SM-G892A Build/NRD90M,4497,7
5,NaN,5937,0
6,NaN,12308,0
7,NaN,12695,0
8,iOS Device,2803,2865
9,NaN,17399,0


In [55]:
validation_data[
    ["DeviceInfo", "card1", "DeviceInfo_unique_card1_count"]
].head(10)

,DeviceInfo,card1,DeviceInfo_unique_card1_count
0,NaN,9300,0
1,F3311,8809,2
2,Windows,10819,4517
3,Windows,9633,4517
4,Windows,17188,4517
5,Windows,9026,4517
6,NaN,6200,0
7,NaN,17188,0
8,NaN,12695,0
9,NaN,16132,0


# Final Features

In [56]:
numeric_features = [
    "TransactionAmt",
    "TransactionDT",
    "log_transaction_amount",
    "addr1_missing",
    "addr2_missing",
    "D7_missing",
    "D12_missing",
    "D13_missing",
    "D14_missing",
    "DeviceInfo_missing",
    "time_since_previous_transaction",
    "has_previous_transaction",
    "card1_transaction_count",
    "card2_transaction_count",
    "addr1_transaction_count",
    "card1_DeviceInfo_transaction_count",
    "card1_unique_DeviceInfo_count",
    "DeviceInfo_unique_card1_count",
]

categorical_features = [
    "ProductCD",
    "card4",
    "card6",
    "DeviceType",
]

In [57]:
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 18
Categorical features: 4


# Feature availability check

In [58]:
selected_features = numeric_features + categorical_features

missing_train = [
    col for col in selected_features
    if col not in train_data.columns
]

missing_validation = [
    col for col in selected_features
    if col not in validation_data.columns
]

print("Missing from train:", missing_train)
print("Missing from validation:", missing_validation)
print("Total selected features:", len(selected_features))

Missing from train: []
Missing from validation: []
Total selected features: 22


In [59]:
for df in [train_data, validation_data]:
    df["log_transaction_amount"] = np.log1p(df["TransactionAmt"])

    for column in [
        "addr1",
        "addr2",
        "D7",
        "D12",
        "D13",
        "D14",
        "DeviceInfo",
    ]:
        df[f"{column}_missing"] = df[column].isna().astype("int8")

In [60]:
simple_features = [
    "log_transaction_amount",
    "addr1_missing",
    "addr2_missing",
    "D7_missing",
    "D12_missing",
    "D13_missing",
    "D14_missing",
    "DeviceInfo_missing",
]

print("Train missing:")
print([c for c in simple_features if c not in train_data.columns])

print("Validation missing:")
print([c for c in simple_features if c not in validation_data.columns])

Train missing:
[]
Validation missing:
[]


In [79]:
train_data["time_since_previous_transaction"] = (train_data["TransactionDT"].diff().fillna(0))
previous_time = train_data["TransactionDT"].iloc[-1]
validation_data["time_since_previous_transaction"] = (validation_data["TransactionDT"].diff())
validation_data.loc[validation_data.index[0],"time_since_previous_transaction"] = (validation_data["TransactionDT"].iloc[0] - previous_time)
train_data["has_previous_transaction"] = (train_data["TransactionDT"].diff().notna().astype("int8"))
validation_data["has_previous_transaction"] = 1

In [80]:
print(train_data[["TransactionDT", "time_since_previous_transaction", "has_previous_transaction"]].head())
print()
print(validation_data[["TransactionDT", "time_since_previous_transaction", "has_previous_transaction"]].head())

   TransactionDT  time_since_previous_transaction  has_previous_transaction
0          86400                              0.0                         0
1          86401                              1.0                         1
2          86469                             68.0                         1
3          86499                             30.0                         1
4          86506                              7.0                         1

   TransactionDT  time_since_previous_transaction  has_previous_transaction
0       12192900                             58.0                         1
1       12192911                             11.0                         1
2       12192913                              2.0                         1
3       12193040                            127.0                         1
4       12193199                            159.0                         1


In [81]:
print(train_data["time_since_previous_transaction"].isna().sum(),validation_data["time_since_previous_transaction"].isna().sum())

0 0


In [64]:
# Final fetaure selected
selected_features = numeric_features + categorical_features

missing_train = [
    col for col in selected_features
    if col not in train_data.columns
]

missing_validation = [
    col for col in selected_features
    if col not in validation_data.columns
]

print("Missing from train:", missing_train)
print("Missing from validation:", missing_validation)
print("Total selected features:", len(selected_features))

Missing from train: []
Missing from validation: []
Total selected features: 22


In [65]:
print("Train missing values:", train_data[selected_features].isna().sum().sum())
print("Validation missing values:", validation_data[selected_features].isna().sum().sum())

Train missing values: 356455
Validation missing values: 96423


# Categorical Encoding


In [66]:
categorical_features = [
    "ProductCD",
    "card4",
    "card6",
    "DeviceType",
]

for column in categorical_features:
    print(f"\n{column}")
    print("Unique:", train_data[column].nunique(dropna=True))
    print("Missing:", train_data[column].isna().sum())
    print("Values:", train_data[column].dropna().unique())


ProductCD
Unique: 5
Missing: 0
Values: ['W' 'H' 'C' 'S' 'R']

card4
Unique: 4
Missing: 833
Values: ['discover' 'mastercard' 'visa' 'american express']

card6
Unique: 4
Missing: 828
Values: ['credit' 'debit' 'debit or credit' 'charge card']

DeviceType
Unique: 2
Missing: 354794
Values: ['mobile' 'desktop']


In [68]:
categorical_features = [
    "ProductCD",
    "card4",
    "card6",
    "DeviceType",
]

for column in categorical_features:
    train_data[column] = train_data[column].fillna("MISSING")
    validation_data[column] = validation_data[column].fillna("MISSING")

In [69]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(train_data[categorical_features])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [70]:
encoder.fit(train_data[categorical_features])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [72]:
X_train_cat = encoder.transform(train_data[categorical_features])
X_validation_cat = encoder.transform(validation_data[categorical_features])

print("Encoded train shape:", X_train_cat.shape)
print("Encoded validation shape:", X_validation_cat.shape)

Encoded train shape: (472432, 18)
Encoded validation shape: (118108, 18)


In [73]:
encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

print("Encoded features:", len(encoded_feature_names))
print("\n".join(encoded_feature_names))

Encoded features: 18
ProductCD_C
ProductCD_H
ProductCD_R
ProductCD_S
ProductCD_W
card4_MISSING
card4_american express
card4_discover
card4_mastercard
card4_visa
card6_MISSING
card6_charge card
card6_credit
card6_debit
card6_debit or credit
DeviceType_MISSING
DeviceType_desktop
DeviceType_mobile


# X_train and X_validation

In [78]:
X_train_num = train_data[numeric_features].copy()
X_validation_num = validation_data[numeric_features].copy()

X_train_cat = pd.DataFrame(X_train_cat,columns=encoded_feature_names,index=train_data.index)

X_validation_cat = pd.DataFrame(X_validation_cat,columns=encoded_feature_names,index=validation_data.index)

X_train = pd.concat([X_train_num, X_train_cat],axis=1)

X_validation = pd.concat([X_validation_num, X_validation_cat],axis=1)

In [75]:
y_train = train_data["isFraud"].copy()
y_validation = validation_data["isFraud"].copy()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)

X_train: (472432, 36)
X_validation: (118108, 36)
y_train: (472432,)
y_validation: (118108,)


In [76]:
print("Missing in X_train:", X_train.isna().sum().sum())
print("Missing in X_validation:", X_validation.isna().sum().sum())
print("Infinite in X_train:", np.isinf(X_train).sum().sum())
print("Infinite in X_validation:", np.isinf(X_validation).sum().sum())

Missing in X_train: 0
Missing in X_validation: 0
Infinite in X_train: 0
Infinite in X_validation: 0


In [77]:
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_validation.mean())

Train fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145


In [82]:
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "model_features"
FEATURE_PATH.mkdir(parents=True, exist_ok=True)

X_train.to_pickle(FEATURE_PATH / "X_train.pkl")
X_validation.to_pickle(FEATURE_PATH / "X_validation.pkl")
y_train.to_pickle(FEATURE_PATH / "y_train.pkl")
y_validation.to_pickle(FEATURE_PATH / "y_validation.pkl")

print("Model features saved successfully.")

Model features saved successfully.


In [83]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)

X_train: (472432, 36)
X_validation: (118108, 36)
y_train: (472432,)
y_validation: (118108,)
